In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("superstore.db")

print("Connection created")

Connection created


In [4]:
df = pd.read_csv("Superstore_raw.csv", encoding="latin1")
df.to_sql(
    "Superstore_raw",conn,if_exists="replace",index=False
)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.head()

Rows: 9994
Columns: 21


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
# Creating customers table

conn.execute("""
CREATE TABLE IF NOT EXISTS customers (

    Customer_ID PRIMARY KEY,
    Customer_Name
)
""")

print("Customers table created successfully!")

# Creating products table

conn.execute("""
CREATE TABLE IF NOT EXISTS products (
    Product_ID PRIMARY KEY,
    Category ,
    Sub_Category ,
    Product_Name 
)
""")

print("Products table created successfully!")

# Creating orders table

conn.execute("""
CREATE TABLE IF NOT EXISTS orders (
    Order_ID ,
    Order_Date ,
    Customer_ID ,
    Product_ID ,
    Sales ,
    Quantity ,
    Profit
)

""")

print("Orders table created successfully!")

Customers table created successfully!
Products table created successfully!
Orders table created successfully!


In [6]:
# Inserting data into customers table

conn.execute("""
INSERT OR IGNORE INTO customers SELECT DISTINCT
    [Customer ID],
    [Customer Name]
FROM Superstore_raw
""")
print("Data inserted into customers table successfully!")

# Inserting data into products table

conn.execute("""
INSERT OR IGNORE INTO products SELECT DISTINCT
    [Product ID],
    Category,
    [Sub-Category],
    [Product Name]
FROM Superstore_raw
""")

print("Data inserted into products table successfully!")

# Inserting data into orders table

conn.execute("""
INSERT INTO orders SELECT DISTINCT
    [Order ID],
    [Order Date],
    [Customer ID],
    [Product ID],
    Sales,
    Quantity,
    Profit
FROM Superstore_raw
""")
conn.commit()

print("Data inserted into orders table successfully!")

Data inserted into customers table successfully!
Data inserted into products table successfully!
Data inserted into orders table successfully!


# Q1. Who are the Top 5 Customers?

In [7]:
df_m1 = pd.read_sql_query("""
SELECT Customer_ID,SUM(Sales) AS Total_Sales
FROM orders
GROUP BY Customer_ID
ORDER BY Total_Sales DESC
LIMIT 5
""", conn)

df_m1

,Customer_ID,Total_Sales
0,SM-20320,250430.50
1,TC-20980,190522.18
2,RB-19360,151173.39
3,TA-21385,145956.20
4,AB-10105,144735.71


## Q2. Who are the Bottom 5 Customers?

In [8]:
df_m2 = pd.read_sql_query("""
SELECT Customer_ID,SUM(Sales) AS Total_Sales
FROM orders
GROUP BY Customer_ID
ORDER BY Total_Sales ASC
LIMIT 5
""", conn)

df_m2

,Customer_ID,Total_Sales
0,TS-21085,48.33
1,LD-16855,53.04
2,CJ-11875,165.20
3,MG-18205,167.39
4,RS-19870,223.28


## Q3. Which Customers Made Only One Order?

In [9]:
df_m3 = pd.read_sql_query("""
SELECT Customer_ID,COUNT(DISTINCT Order_ID) AS Total_Orders
FROM orders
GROUP BY Customer_ID
HAVING COUNT(DISTINCT Order_ID) = 1
""", conn)

df_m3

,Customer_ID,Total_Orders
0,AO-10810,1
1,AR-10570,1
2,CJ-11875,1
3,JC-15385,1
4,JR-15700,1
5,LD-16855,1
6,MG-18205,1
7,PH-18790,1
8,RE-19405,1
9,RM-19750,1


## Q4. Which Customers Have Above-Average Sales?

In [10]:
df_m4 = pd.read_sql_query("""
WITH customer_sales AS(SELECT Customer_ID,SUM(Sales) AS Total_Sales
FROM orders
GROUP BY Customer_ID)

SELECT * FROM customer_sales
WHERE Total_Sales >(SELECT AVG(Total_Sales)
FROM customer_sales)
ORDER BY Total_Sales DESC
""", conn)

df_m4

,Customer_ID,Total_Sales
0,SM-20320,250430.50
1,TC-20980,190522.18
2,RB-19360,151173.39
3,TA-21385,145956.20
4,AB-10105,144735.71
...,...,...
289,JK-16120,29324.84
290,SW-20455,29215.44
291,ML-17410,29215.00
292,RD-19585,29128.94


## Q5. What is the Highest Order Value Per Customer?

In [11]:
df_m5 = pd.read_sql_query("""
SELECT Customer_ID,MAX(Sales) AS Highest_Order_Value
FROM orders
GROUP BY Customer_ID
ORDER BY Highest_Order_Value DESC
""", conn)

df_m5

,Customer_ID,Highest_Order_Value
0,SM-20320,22638.480
1,TC-20980,17499.950
2,RB-19360,13999.960
3,TA-21385,11199.968
4,HL-15040,10499.970
...,...,...
788,CJ-11875,16.520
789,MG-18205,12.320
790,RS-19870,9.648
791,LD-16855,5.304


In [12]:
# Closing the SQLite connection

conn.close()

print("Database connection closed")

Database connection closed
